In [1]:
import os

os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"

In [2]:
from pathlib import Path

import nibabel as nib

image_path = Path("/scratch/imaging/ds005237/sub-NDARINVXP963GZ5/func/sub-NDARINVXP963GZ5_task-restPA_run-01_bold.nii.gz")

In [3]:
import json

import numpy as np

sidecar = json.loads(
    (image_path.parent / image_path.name.removesuffix(".gz").removesuffix(".nii"))
    .with_suffix(".json")
    .read_text()
)
slice_timing = np.asanyarray(sidecar["SliceTiming"])

In [4]:
import scipy.stats

slice_ranks = scipy.stats.rankdata(
    (slice_timing * 10000).astype(int), method="dense"
).astype(int)

In [5]:
frames_per_second = np.reciprocal(
    sidecar["RepetitionTime"] / np.unique(slice_ranks).size
)
frames_per_second

np.float64(11.25)

In [6]:
volume_count = round(60 / sidecar["RepetitionTime"])
volume_count

75

In [7]:
slice_timing_path = Path("/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5_task-restPA_run-01_bold_valid_tshift.nii.gz")
image = nib.nifti1.load(slice_timing_path) # pyright: ignore[reportAttributeAccessIssue]

In [8]:
data = image.get_fdata()
data.shape

(104, 104, 72, 488)

In [25]:
import nitransforms
from fmriprep.interfaces.resampling import load_transforms, reconstruct_fieldmap
from sdcflows.transform import B0FieldTransform
from sdcflows.utils.tools import ensure_positive_cosines

derivatives_path = Path("/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5")

motion_correction_transforms = nitransforms.linear.load(
    derivatives_path
    / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_from-orig_to-boldref_mode-image_desc-hmc_xfm.txt"
)
if not isinstance(motion_correction_transforms, nitransforms.linear.LinearTransformsMapping):
    raise TypeError

coeffs = nib.nifti1.load(  # pyright: ignore[reportAttributeAccessIssue]
    derivatives_path
    / "fmap/sub-NDARINVXP963GZ5_fmapid-SpinEchoFieldMapfmap_desc-coeff_fieldmap.nii.gz"
)
boldref = nib.nifti1.load(  # pyright: ignore[reportAttributeAccessIssue]
    derivatives_path
    / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_desc-hmc_boldref.nii.gz"
)
fmap_reference = nib.nifti1.load(  # pyright: ignore[reportAttributeAccessIssue]
    derivatives_path
    / "fmap/sub-NDARINVXP963GZ5_fmapid-SpinEchoFieldMapfmap_desc-epi_fieldmap.nii.gz"
)
target, _ = ensure_positive_cosines(boldref)

b0_field_transform = B0FieldTransform(coeffs=coeffs)  # pyright: ignore[reportCallIssue]
b0_field_transform.mapped = reconstruct_fieldmap(
    coefficients=[coeffs],
    fmap_reference=fmap_reference,
    target=target,
    transforms=load_transforms(
        [
            derivatives_path
            / "func/sub-NDARINVXP963GZ5_task-restPA_run-01_from-boldref_to-SpinEchoFieldMapfmap_mode-image_desc-fmap_xfm.txt"
        ],
        [True],
    ),
)

pe_dir = sidecar["PhaseEncodingDirection"]
flipped_pe = pe_dir[0] if pe_dir.endswith("-") else pe_dir + "-"
mapped = b0_field_transform.mapped
forward_displacement_image = b0_field_transform.to_displacements(
    ro_time=sidecar["TotalReadoutTime"],
    pe_dir=pe_dir,
    itk_format=True,
)
forward_displacement_image.to_filename("forward_displacement.nii.gz")

In [26]:
import ants

# 1. Load the reference image
boldref_path = "/scratch/imaging/ds005237-fmriprep/sub-NDARINVXP963GZ5/func/sub-NDARINVXP963GZ5_task-restPA_run-01_desc-hmc_boldref.nii.gz"
boldref_img = ants.image_read(boldref_path)

# forward_field = ants.image_read("forward_displacement.nii.gz")
# inverse_field_initial_estimate = forward_field.new_image_like(  # pyright: ignore[reportAttributeAccessIssue]
#     forward_field.numpy() * -1.0
# )
# backward_field = ants.invert_displacement_field(
#     forward_field, inverse_field_initial_estimate, maximum_number_of_iterations=1_000
# )
# backward_path = Path("backward_displacement.nii.gz")
# ants.image_write(backward_field, backward_path.as_posix())

warped_boldref = ants.apply_transforms(
    fixed=boldref_img,
    moving=boldref_img,
    transformlist=["forward_displacement.nii.gz"],
    whichtoinvert=[False],
    interpolator="linear",
)

ants.image_write(warped_boldref, "sanity_check_warped_boldref.nii.gz")

In [32]:
volume = nib.Nifti1Image(data[..., 0], image.affine, image.header)
volume.to_filename("volume.nii.gz")

volume_img = ants.image_read("volume.nii.gz")

hmc_xfm = motion_correction_transforms[0]
hmc_path = Path("motion_correction_transform.mat")
hmc_xfm.to_filename(hmc_path, fmt="itk")

warped_boldref = ants.apply_transforms(
    fixed=volume_img,
    moving=boldref_img,
    transformlist=[
        "forward_displacement.nii.gz",
        hmc_path.as_posix(),
    ],
    whichtoinvert=[False, False],
    interpolator="linear",
)

ants.image_write(warped_boldref, "sanity_check_warped_boldref.nii.gz")